**Predicting the selling price of a car using historical data.**

### **STEP 1 — Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### **STEP 2 — Load Dataset | Understanding**

In [ ]:
df = pd.read_csv("cardetails.csv")
df.head(3)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

### **STEP 3 — Data Cleaning**

In [ ]:
# If the null values were less, I would have filled up with fillna.
# df.fillna(method='ffill', inplace=True)

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
# Ealier it was (8128, 13) after removing null.

df.shape

In [ ]:
df.drop(columns=["torque"], inplace=True)

In [ ]:
df.head(1)

In [ ]:
# Duplicated check

df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

In [ ]:
for col in df.columns:
  print(f"Unique Values of + {col}")
  print(df[col].unique())
  print("===========================================\n")

In [ ]:
def get_brand_names(x):
  return x.split(" ")[0]

In [ ]:
df["name"] = df["name"].apply(get_brand_names)

In [ ]:
df["mileage"] = df["mileage"].apply(get_brand_names)

In [ ]:
df["engine"] = df["engine"].apply(get_brand_names)

In [ ]:
df["max_power"] = df["max_power"].apply(get_brand_names)

In [ ]:
df["name"].unique()

In [ ]:
df["name"].nunique()

In [ ]:
df["mileage"].unique()

In [ ]:
df["engine"].unique()

In [ ]:
df["max_power"].unique()

In [ ]:
for col in df.columns:
  print(f"Unique Values of + {col}")
  print(df[col].unique())
  print("===============================================\n")

In [ ]:
df.drop(columns=["name"], inplace=True)

In [ ]:
df.info()

In [ ]:
df.head(3)

In [ ]:
df["selling_price"] = df["selling_price"].astype(int)

In [ ]:
df["km_driven"] = df["km_driven"].astype(int)

In [ ]:
df["mileage"] = df["mileage"].astype(float)

In [ ]:
df["engine"] = df["engine"].astype(int)

In [ ]:
df["max_power"] = df["max_power"].astype(float)

In [ ]:
df["seats"] = df["seats"].astype(int)

In [ ]:
df.info()

In [ ]:
df.head(3)

### **STEP 4 — Feature Engineering.**

Model does not understand anything from **year** column only therefore creating **car_age** with the help of feature engineering so that model can understands better.


In [ ]:
current_year = 2026
df["car_age"] = current_year - df["year"]


In [ ]:
df.drop(columns=["year"], inplace=True)

In [ ]:
df.head(1)

### **STEP 5 — EDA**

In [ ]:
!pip install ydata-profiling

In [ ]:
from ydata_profiling import ProfileReport
profile = ProfileReport(df)
profile.to_file(output_file="output.html")

In [ ]:
from google.colab import files
files.download("output.html")

**Scientific notation (1e6) / normalized scale**

Matplotlib automatically does this when numbers are large (like selling_price)

100000 → shown as 0.1 (with 1e6 multiplier)




> **plt.ticklabel_format(style='plain', axis='y')**



In [ ]:
sns.histplot(df["selling_price"], kde=True)
plt.title("Distribution of Selling Price")
plt.ticklabel_format(style='plain', axis='x')
plt.show()

**Insight:**

The selling price distribution is right-skewed,

indicating most cars are in the lower price range

with few expensive outliers.

In [ ]:
plt.figure(figsize=(6,4))
sns.scatterplot(x=df["car_age"], y=df["selling_price"])
plt.xlabel("Car Age")
plt.ylabel("Selling Price")
plt.title("Car Age vs Selling Price")
plt.ticklabel_format(style='plain', axis='y')
plt.show()

**Insight:**

There is a negative relationship between car age

and price-older cars age increases then selling

price becomes lower.

In [ ]:
plt.figure()
sns.scatterplot(x=df["engine"], y=df["selling_price"])
plt.xlabel("Engine")
plt.ylabel("Selling Price")
plt.title("Engine vs Selling Price")
plt.ticklabel_format(style='plain', axis='y')
plt.show()


**Insight:**

Engine size shows a positive correlation with price,

 meaning higher engine capacity cars tend to be more expensive.

In [ ]:
sns.scatterplot(x=df["max_power"], y=df["selling_price"])
plt.title("Max Power vs Selling Price")
plt.ticklabel_format(style='plain', axis='y')
plt.show()

**Insight:**

Cars with higher max power generally have higher selling prices.

In [ ]:
sns.boxplot(x=df["fuel"], y=df["selling_price"])
plt.title("Fuel vs Selling Price")
plt.ticklabel_format(style='plain', axis='y')
plt.show()

**Insight:**

Diesel cars tend to have slightly higher median prices compared to petrol cars.

In [ ]:
sns.boxplot(x=df["transmission"], y=df["selling_price"])
plt.title("Transmission vs Selling Price")
plt.ticklabel_format(style='plain', axis='y')
plt.show()

**Insight:**

Automatic cars generally have higher prices compared to manual cars

In [ ]:
plt.figure()
sns.heatmap(df.corr(numeric_only=True), annot=False)
plt.title("Correlation Heatmap")
plt.show()


**Insight:**


Correlation analysis is only applicable to numerical features,
so I filtered numeric columns before generating the heatmap.

Features like engine and max_power show strong positive correlation with price,

while car_age shows negative correlation

### **STEP 6 — Visualize Outliers | Remove**

In [ ]:
plt.figure()
sns.boxplot(x=df["selling_price"])
plt.title("Selling Price")
plt.ticklabel_format(style="Plain", axis="x")
plt.show()

**Note:**

I first perform EDA to identify outliers visually, then

decide whether to remove or cap them based on business logic.

In [ ]:
print(df["selling_price"].min())
print(df["selling_price"].max())
print(df["selling_price"].mean())

In [ ]:
df.describe()

In [ ]:
Q1 = df["selling_price"].quantile(0.25)
Q3 = df["selling_price"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR


print(lower_limit, upper_limit)

In [ ]:
# To see how many outliers exist in the data

outliers = df[(df["selling_price"] < lower_limit) | (df["selling_price"] > upper_limit)]
print(f"Total number of outliers: {len(outliers)}")

In [ ]:
df = df[(df['selling_price'] >= lower_limit) & (df['selling_price'] <= upper_limit)]

In [ ]:
plt.figure()
sns.boxplot(x=df["selling_price"])
plt.title("Selling Price")
plt.ticklabel_format(style="Plain", axis="x")
plt.show()

In [ ]:
print(df["selling_price"].min())
print(df["selling_price"].max())
print(df["selling_price"].mean())

### **STEP 7 — Encoding (Categorical → Numeric)**

In [ ]:
df = pd.get_dummies(df, drop_first=True)

In [ ]:
df.head()

### **STEP 8 — Feature Selection**

In [ ]:
# Seeing correlation with target - Check Correlation First

corr = df.corr()['selling_price'].sort_values(ascending=False)
print(corr)

In [ ]:
# Visualize Correlation (Heatmap)

plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
df = df.drop(columns=["mileage", "seller_type_Trustmark Dealer", "fuel_LPG",
                      "owner_Fourth & Above Owner"])

print("Remaining columns:", df.columns.tolist())

In [ ]:
x = df.drop(columns=["selling_price"])
y = df["selling_price"]

In [ ]:
x.shape

In [ ]:
y.shape

### **STEP 9 — Train-Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
print(y_train.shape)
print(y_test.shape)

### **STEP 10 — Model Building (Random Forest)**


In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
model = RandomForestRegressor()
model.fit(X_train, y_train)

### **STEP 11 — Prediction**

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
print(f"Predicted: {y_pred[:5]}")
print(f"Actual {y_test[:5].values}")

### **STEP 12 — Model Evaluation**

**Car price depends on:**

> → Engine + Max Power + Car Age

> → Engine + Max Power + Car Age

> → ALL TOGETHER at the same time

> → Not a simple straight line

> → Random Forest handles this well



In [ ]:
from sklearn.metrics import r2_score

print("R2 Score:", r2_score(y_test, y_pred))

Why Random Forest?
→ "Because my data is complex and
   Linear Regression only gave 0.67
   Random Forest gave 0.87"




---


Why drop those columns?
→ "Because their correlation was
   close to 0, meaning they had
   no relationship with selling price"



---


What is R2 Score?
→ "It tells how accurate my model is
   My model is 87% accurate" ✅

### **STEP 13 — Visualization**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, color='blue', alpha=0.5)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         color='red', linewidth=2)
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted Price')
plt.show()

### **STEP 14 — Model Interpretation**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
)

importance.sort_values(ascending=False)
print(importance.sort_values(ascending=False))


In [ ]:
importance.sort_values().plot(
    kind='barh',
    figsize=(10, 6),
    color='green'
)
plt.title('Feature Importance')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

### **STEP 15 — Save Model**

In [ ]:
import pickle

# Save the model
with open('car_price_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model Saved Successfully!")

In [ ]:
"""# Load the model
with open('car_price_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

print("Model Loaded Successfully!")

# Use it to predict
y_pred_new = loaded_model.predict(X_test)
print("Predictions:", y_pred_new[:5])"""

### **STEP 16 — Conclusion**


I wanted to predict the selling price
of a used car based on its features
like engine, max power, car age etc.

> **STEP 1** - Import Libraries

> **STEP 2** - Load Dataset | Understanding

> **STEP 3** - Data Cleaning

> **STEP 4** - Feature Engineering (car_age)

> **STEP 5** - EDA

> **STEP 6** - Visualize Outliers | Remove

> **STEP 7** - Encoding (Categorical - Numeric)

> **STEP 8** - Feature Selection (correlation)

> **STEP 9** - Train Test Split (80/20)

> **STEP 10** - Model Building (Random Forest) → 0.87

> **STEP 11** - Prediction

> **STEP 11** - Model Evaluation

> **STEP 13** - Visualization

> **STEP 14** - Model Interpretation

> **STEP 15** - Save Model

> **STEP 16** - Conclusion